# Lab 4 — Linear Systems: Let It Swing, Shake It, Steer It

**ROB7104 / ROB8104 Mathematics for Robotics · passive, forced and controlled linear systems**

This notebook is a reading. There is no code to run and nothing to hand in. Every idea comes with a short video of a simulated robot, and every video has a live version on the lab's web page where you can turn the camera and move the sliders yourself:

**Interactive page:** [realgaule.github.io/rob7104/lab4](https://realgaule.github.io/rob7104/lab4/)

It follows Lecture 9 (linear systems and stability), Lecture 3 (eigenvalues and normal modes), Lecture 10 (Lyapunov's method, the Kalman filter) and the slides on controllability, observability and rank.

> If a video does not play inside your notebook viewer, open `Lab4.html` (same folder) in a browser.

## 0. One equation: $\dot x = Ax + g(t)$

<video controls loop muted playsinline width="100%" src="media/hero.mp4" poster="media/hero_poster.png"></video>

The examples in this lab:

| Robot | What swings | Where it appears |
|---|---|---|
| A **drone** carrying a two-part slung load | a heavy crate (5 kg) on a 0.9 m cable, and a light parcel (1 kg) 0.7 m below it | Parts 1 and 2 |
| A **robot arm** carrying a glass of water | the water surface | Parts 2 and 3 |
| An overhead **gantry crane** | loads hanging from its trolley | Part 3 |

They look different, but they are the same kind of system: **a support that moves sideways, with something that swings below it.** The support is the drone, the hand, or the trolley. Its horizontal acceleration $a(t)$ is the only way the outside world acts on what swings.

### 0.1 From the robot to $\dot x = Ax + g(t)$

Take the drone's load. Let $\theta_1$ and $\theta_2$ be the angles of the upper and lower cable from the vertical. Lagrange's equations give two coupled nonlinear equations (the double pendulum of Lecture 9). For **small angles and slow motion** we keep only the terms of first order, $\sin\theta \approx \theta$, $\cos\theta \approx 1$, and drop products such as $\dot\theta^2\theta$. What is left is linear:

$$
M\ddot\theta + C\dot\theta + K\theta = -h\,a(t),
\qquad \theta = \begin{pmatrix}\theta_1\\ \theta_2\end{pmatrix},
$$

$$
M = \begin{pmatrix}(m_1+m_2)\,l_1^2 & m_2 l_1 l_2\\ m_2 l_1 l_2 & m_2 l_2^2\end{pmatrix},\quad
K = \begin{pmatrix}(m_1+m_2)\,g\,l_1 & 0\\ 0 & m_2\,g\,l_2\end{pmatrix},\quad
h = \begin{pmatrix}(m_1+m_2)\,l_1\\ m_2 l_2\end{pmatrix}.
$$

$M$ and $K$ are symmetric. $C$ collects friction at the two hinges. The right-hand side is the pull that the load feels when its support accelerates.

Now stack angles and angular velocities into one **state vector** $x = (\theta_1, \theta_2, \dot\theta_1, \dot\theta_2)^\top$. The second-order equation becomes a first-order one:

$$
\dot x = Ax + g(t),\qquad
A = \begin{pmatrix} 0 & I\\ -M^{-1}K & -M^{-1}C \end{pmatrix},\qquad
g(t) = \begin{pmatrix} 0\\ -M^{-1}h \end{pmatrix} a(t).
$$

The water in the glass and the crane's loads lead to the same form, with their own $A$.

### 0.2 The one formula of this lab

The solution of $\dot x = Ax + g(t)$ with $x(0) = x_0$ is

$$
\boxed{\;x(t) \;=\; e^{At}x_0 \;+\; \int_0^t e^{A(t-\tau)}\,g(\tau)\,d\tau\;}
$$

The first term is what the system does by itself. The second is what the forcing adds. The three parts of the lab are three choices of $g$:

| Part | Forcing | Question |
|---|---|---|
| **1. Passive** | $g = 0$ | What does $e^{At}$ do? |
| **2. Forced** | $g(t)$ is given by the world | What does the integral do? |
| **3. Controlled** | $g = Bu$ and we choose $u$ | Which states can we reach, and how cheaply? |

**A word on what you are watching.** In every video the solid robot is a full nonlinear simulation (MuJoCo), not the linear model. The linear model appears as translucent "ghosts" or in the flat diagrams. So each video is also a small experiment: the theory predicts, the robot checks.

---

# Part 1 — Passive systems: $\dot x = Ax$

The drone hovers. Nothing pushes the load. We pull it aside and let go.

## 1.1 Normal modes and the matrix exponential

**Question.** Released from an arbitrary position, the load moves in a complicated way. Is there a simple description?

**Idea.** Look for motions in which both cables swing *in step* at one frequency: $\theta(t) = v\cos(\omega t)$ with a fixed shape $v$. Without friction, substituting into $M\ddot\theta + K\theta = 0$ gives

$$
K v = \omega^2 M v .
$$

This is an eigenvalue problem (Lecture 3). A system with two angles has two solutions, the **normal modes**. For this load:

| | Frequency | Period | Shape $(\theta_1 : \theta_2)$ | What it looks like |
|---|---|---|---|---|
| Mode 1 | 2.93 rad/s | 2.14 s | $(0.49 : 1)$ | both cables lean the same way |
| Mode 2 | 4.61 rad/s | 1.36 s | $(0.27 : -1)$ | the parcel swings against the crate |

Because the equation is linear, a sum of solutions is a solution. So **every** small free motion is

$$
\theta(t) = c_1 v_1\cos(\omega_1 t + \varphi_1) + c_2 v_2\cos(\omega_2 t + \varphi_2).
$$

<video controls loop muted playsinline width="100%" src="media/s11_modes.mp4" poster="media/s11_modes_poster.png"></video>

*Left: released in the shape of mode 1 (12°). Middle: mode 2 (8°). Right: an arbitrary release. The faint green and red loads are its two modal parts; the yellow robot follows their sum.*

> **Takeaway.** Any small swing is a mix of two pure swings.
> ▶ Try it: [Normal modes demo](https://realgaule.github.io/rob7104/lab4/#d11)

### The same thing, said with $e^{At}$

In state form the free motion is $\dot x = Ax$, and its solution is $x(t) = e^{At}x_0$, where the **matrix exponential** is defined by the same series as the ordinary one (Lecture 9):

$$
e^{At} = I + At + \frac{(At)^2}{2!} + \frac{(At)^3}{3!} + \cdots
$$

For each time $t$, $e^{At}$ is just a matrix: a linear map that takes *where you started* to *where you are now*. It moves all initial conditions at once. If $A = S\Lambda S^{-1}$ is diagonalizable then $e^{At} = S\,e^{\Lambda t}S^{-1}$, so the eigenvalues of $A$ decide everything. Here they are $\pm i\omega_1$ and $\pm i\omega_2$: the normal modes again.

The next video shows this for a single pendulum (one cable, 1.2 m), whose state $(\theta, \dot\theta)$ fits in a plane. Twenty-five loads hang from one drone, each started differently. On the right, each load is one point. Lab 1 showed a matrix acting on a grid; here the matrix changes continuously with time.

<video controls loop muted playsinline width="100%" src="media/s11b_expm.mp4" poster="media/s11b_expm_poster.png"></video>

*First half, no friction: the grid turns rigidly. Second half, with friction (damping ratio 0.08): it spirals inward. Straight lines stay straight throughout, because $e^{At}$ is linear. The plane is drawn in the coordinates $(\theta,\ \dot\theta/\omega)$, in which the frictionless motion is a pure rotation.*

The white circle of initial conditions is mapped to an ellipse. Its longest semi-axis, relative to the circle's radius, is the **matrix norm** $\lVert e^{At}\rVert_2$ (in these coordinates): the largest factor by which any initial condition has been stretched. We will need it in 1.3.

> **Takeaway.** $e^{At}$ moves every initial condition at once, and it is a linear map at every instant.
> ▶ Try it: [The flow $e^{At}$](https://realgaule.github.io/rob7104/lab4/#d11b)

## 1.2 When the linear model stops being right

**Question.** We threw terms away to get a linear model. When does that matter?

The video releases the load with both cables at 5°, 30° and 90°. The solid load is the robot. The blue ghost is the linear model started from the same position. The yellow number is the distance between the two parcels.

<video controls loop muted playsinline width="100%" src="media/s12_linearize.mp4" poster="media/s12_linearize_poster.png"></video>

| Release angle | Largest gap in 12 s | When the gap first exceeds 10 cm |
|---|---|---|
| 5° | 0.35 cm | never |
| 30° | 84 cm | after 1.5 s |
| 90° | 304 cm | after 0.2 s |

At 5° the model and the robot are indistinguishable. At 30° the model is good for about a second and then drifts: its frequencies are slightly wrong at this amplitude, and a small error in frequency grows into a large error in position. At 90° the real parcel is whipped over the top by the heavy crate after 1.2 s, something a linear model cannot do at all.

Two things are worth remembering.

- "Small" depends on the system and on how long you watch. For this load, mode 1 released at 12° stays within 2% of the linear model for 6 s; mode 2 released at 8° drifts by 9% in the same time.
- Everything else in this lab is linear theory. It is exact for the model and approximate for the robot. Where the difference is visible, the notebook says so.

> **Takeaway.** The linear model is right for small swings only, and its errors grow with time.
> ▶ Try it: [When linear fails](https://realgaule.github.io/rob7104/lab4/#d12)

## 1.3 Stability: friction moves the eigenvalues to the left

**Question.** Will the swing die out, and how fast?

With friction at the hinges, $C \neq 0$ and the eigenvalues of $A$ leave the imaginary axis. An eigenvalue $\lambda = -\sigma \pm i\omega_d$ contributes a term $e^{-\sigma t}\cos(\omega_d t + \varphi)$ to the motion: it oscillates at $\omega_d$ and decays at the rate $\sigma$. So:

> $\dot x = Ax$ is **asymptotically stable** (every solution tends to zero) exactly when every eigenvalue of $A$ has negative real part.

<video controls loop muted playsinline width="100%" src="media/s13_stability.mp4" poster="media/s13_stability_poster.png"></video>

*Three drones release the same swing with no, light and heavy hinge friction. Top right: the eigenvalues of $A$ for the three cases. Bottom right: $\lVert e^{At}\rVert_2$ on a logarithmic axis.*

| Friction | Eigenvalues of $A$ | Damping ratios (mode 1, mode 2) |
|---|---|---|
| None | $\pm 2.93i,\ \pm 4.61i$ | 0, 0 |
| Light | $-0.17 \pm 2.94i,\ -0.53 \pm 4.57i$ | 0.06, 0.12 |
| Heavy | $-0.79 \pm 2.97i,\ -2.73 \pm 3.46i$ | 0.26, 0.62 |

**A bound that needs no solution.** Lecture 9 measures the size of $e^{At}$ by a matrix norm. If all eigenvalues have negative real part, there are constants $c \ge 1$ and $a > 0$ with

$$
\lVert e^{At}\rVert \le c\,e^{-at}\quad (t \ge 0),
\qquad\text{so}\qquad \lVert x(t)\rVert \le c\,e^{-at}\lVert x_0\rVert .
$$

Any $a$ smaller than the slowest decay rate works. In the video the dashed lines are such bounds, computed numerically: $7.08\,e^{-0.15t}$ for light friction and $4.60\,e^{-0.70t}$ for heavy friction. Note that $c > 1$: the norm can grow before it decays, because the state mixes angles and velocities and energy moves between them.

**The same conclusion from energy.** Lyapunov's direct method (Lecture 10) reaches stability without eigenvalues. The energy $V = \tfrac12\dot\theta^\top M\dot\theta + \tfrac12\theta^\top K\theta$ is positive, and along solutions $\dot V = -\dot\theta^\top C\dot\theta \le 0$: friction can only remove energy.

> **Takeaway.** Stable means every eigenvalue is in the left half-plane; the one closest to the imaginary axis sets how slowly the motion dies.
> ▶ Try it: [Stability](https://realgaule.github.io/rob7104/lab4/#d13)

---

# Part 2 — Forced systems: $\dot x = Ax + g(t)$

Now the support moves, and we do not get to choose how. A robot arm jerks a glass of water; a gust shakes the drone.

## 2.1 Any response is a sum of responses to small kicks

**Where the formula comes from.** Multiply $\dot x - Ax = g$ by $e^{-At}$. The left side becomes a total derivative, $\tfrac{d}{dt}\big(e^{-At}x\big) = e^{-At}g(t)$. Integrate from $0$ to $t$ and multiply by $e^{At}$:

$$
x(t) = e^{At}x_0 + \int_0^t e^{A(t-\tau)}\,g(\tau)\,d\tau .
$$

**How to read it.** Cut time into short slices. During the slice around time $\tau$ the forcing gives the state a small kick $g(\tau)\,d\tau$. After that, the kick evolves freely, like an initial condition: at time $t$ it has become $e^{A(t-\tau)}g(\tau)\,d\tau$. The integral adds up the kicks from all earlier slices. This works because the system is linear: responses add.

**The robot.** A Franka arm carries a glass of water 30 cm sideways in 0.93 s and stops. The water is not simulated as a fluid. Its first sloshing mode behaves like a pendulum, so we model the tilt $\theta$ of the surface by

$$
\ddot\theta + 2\zeta\omega\,\dot\theta + \omega^2\theta = -\frac{a(t)}{l},\qquad \omega = 22.7\ \text{rad/s (period 0.28 s)},\quad \zeta = 0.02,\quad l = g/\omega^2 = 1.9\ \text{cm},
$$

where $a(t)$ is the hand's acceleration. The video is slowed down five times.

<video controls loop muted playsinline width="100%" src="media/s21_kicks.mp4" poster="media/s21_kicks_poster.png"></video>

*Top right: the hand's acceleration, cut into 15 slices. Middle: the slosh that each slice starts on its own. Bottom: their running sum (yellow) lies on the tilt of the linear model.*

The tilt peaks at 17.8° on the nonlinear pendulum and 18.0° on the linear model, and the two never differ by more than 0.5°. The hand stops at 0.93 s, but the water does not: each kick keeps ringing, and the kicks from braking do not happen to cancel those from pushing. The first swing after the stop is still 12°.

> **Takeaway.** The response to any motion is the sum of the responses to its small kicks.
> ▶ Try it: [A sum of kicks](https://realgaule.github.io/rob7104/lab4/#d21): move the hand yourself and try not to spill.

## 2.2 Sustained forcing: bounded response, resonance, and a steady state that forgets

**Question.** If the forcing never stops, does the response stay bounded?

**A bound from the formula.** Suppose the passive system is stable, $\lVert e^{At}\rVert \le c\,e^{-at}$ (Part 1.3), and the forcing is bounded, $\lVert g(t)\rVert \le \gamma$. Taking norms in the formula of 2.1, as in Lecture 9,

$$
\lVert x(t)\rVert \;\le\; c\,e^{-at}\lVert x_0\rVert + c\gamma\int_0^t e^{-a(t-\tau)}d\tau \;\le\; c\,e^{-at}\lVert x_0\rVert + \frac{c\,\gamma}{a}.
$$

The first term dies out. The second is a ceiling that does not depend on where the system started. **Bounded input, bounded state.** But the ceiling is proportional to $1/a$: a weakly damped system can have a very large bounded response.

**Resonance.** Shake the drone sideways at one frequency, $a(t) = a_0\cos(\omega t)$. After the transient, the load swings at the same frequency $\omega$, with an amplitude that depends on $\omega$. Writing $g(t) = \mathrm{Re}\,(\hat g\,e^{i\omega t})$, the steady motion is $x_{ss}(t) = \mathrm{Re}\,\big((i\omega I - A)^{-1}\hat g\,e^{i\omega t}\big)$. The matrix $(i\omega I - A)^{-1}$ is large when $i\omega$ is close to an eigenvalue of $A$, that is, when the shaking frequency is close to a natural frequency.

<video controls loop muted playsinline width="100%" src="media/s22_resonance.mp4" poster="media/s22_resonance_poster.png"></video>

*Seven drones, each shaken sideways by the same ±1.6 cm, at seven frequencies (labelled as multiples of the mode-1 frequency $\omega_1$). The sliding dot above each drone shows its shake, drawn 12 times larger. Bottom: the steady swing predicted by the linear model (rings) and the swing measured on the robot (dots). Played twice as fast.*

| Shake frequency | Lower cable, linear model | Lower cable, robot |
|---|---|---|
| $0.5\,\omega_1$ | 0.3° | 0.3° |
| $1.0\,\omega_1$ (mode 1) | 14.9° | 15.0° |
| $1.2\,\omega_1$ | 6.3° | 6.3° |
| $1.4\,\omega_1$ (upper cable vertical) | 7.4° | 7.5° |
| $1.57\,\omega_1$ (mode 2) | 12.5° | 12.0° |
| $2.0\,\omega_1$ | 1.8° | 1.8° |
| $3.0\,\omega_1$ | 0.4° | 0.4° |

The same small shake gives a swing fifty times larger at mode 1 than at half that frequency. Each resonant drone swings in the *shape* of its mode: at mode 1 both cables lean together, at mode 2 the parcel swings against the crate.

The blue drone shows something less obvious. At $\omega^2 = (m_1+m_2)g/(m_1 l_2)$, about $1.4\,\omega_1$, the upper cable's response has a zero (an **anti-resonance**): the upper cable stays vertical under the drone and only the lower cable swings. The heavy crate is not at rest; it rides along with the drone. The linear model predicts this exactly without friction, and with the small friction used here the upper cable swings 0.24° while the lower one swings 7.4°. A second special frequency, $\omega^2 = g/l_2$ (about $1.28\,\omega_1$), does something else: there the crate itself stands almost still in space while the drone moves above it. The interactive demo has a button for each.

The robot follows the linear curve within 1% everywhere except at mode 2, where it swings 3.6% less: the faster mode is also the more nonlinear one.

<video controls loop muted playsinline width="100%" src="media/s22_sweep.mp4" poster="media/s22_sweep_poster.png"></video>

*One drone, shaken with a frequency that rises from $0.8\,\omega_1$ to $1.85\,\omega_1$. Yellow: the swing measured on the robot. White: the steady swing of the linear model at each frequency. Played six times as fast.*

The swing rises through mode 1, falls, the upper cable goes quiet near the anti-resonance, and the swing rises again through mode 2. The yellow trace lags behind the white curve: it peaks about 4% later in frequency. That is not a model error. A lightly damped system needs several time constants (here about 7 s for mode 1) to build up its steady swing, and the sweep moves on before it can. Sweeping 30 times more slowly would bring the trace onto the curve.

**The steady state forgets the start.** In $x(t) = e^{At}x_0 + \int_0^t\!\cdots$, the initial condition appears only in the first term, and that term decays. Two copies of the same stable system, driven by the same forcing from different starts, differ by $e^{At}(x_0 - x_0')$, which tends to zero.

<video controls loop muted playsinline width="100%" src="media/s22_forget.mp4" poster="media/s22_forget_poster.png"></video>

*Five loads, released from very different positions under one drone that is shaken ±10 cm at $2\,\omega_1$. Top left: the largest distance between any two of them (logarithmic bar).*

The gap between the loads is 1.9 m at the start, under 10 cm after 6.6 s, and under 1 cm after 11.6 s. From then on the five move as one, swinging 12° in step with the shaking. The gap shrinks at about the rate of the slowest decaying eigenvalue of $A$, as the formula predicts. (It pulses a little on the way down, because the five starts were all released at rest and their leftover free motions swing in phase.)

> **Takeaway.** A stable system answers a bounded forcing with a bounded motion that forgets its start; shake it at a natural frequency and that mode grows large.
> ▶ Try it: [Resonance](https://realgaule.github.io/rob7104/lab4/#d22)

---

# Part 3 — Controlled systems: $\dot x = Ax + Bu$

Now we choose the forcing. The robot is an overhead gantry: a trolley on a rail, with loads hanging from it. The **input** $u(t)$ is the trolley's acceleration. The trolley by itself is the double integrator $\dot p = v,\ \dot v = u$.

## 3.1 Controllability and observability

### What one motor can reach

**Question.** There is one motor and there are many states. Can a suitable $u(t)$ take the system from any state to any other?

If yes, the system is **controllable**. For $\dot x = Ax + Bu$ with $n$ states there is a test that needs no simulation:

$$
\mathcal R = \begin{pmatrix} B & AB & A^2B & \cdots & A^{n-1}B\end{pmatrix},\qquad \text{controllable} \iff \operatorname{rank}\mathcal R = n .
$$

$B$ says where the input enters directly; $AB$, $A^2B$, … say where the dynamics carry it next. For the bare trolley, $\mathcal R = \begin{pmatrix}0&1\\1&0\end{pmatrix}$ has rank 2: acceleration changes velocity, and velocity changes position.

**A twin hoist.** Hang two separate loads from the trolley, on cables of lengths $l_1$ and $l_2$. For small angles each one obeys

$$
\ddot\theta_k = -\frac{g}{l_k}\,\theta_k - \frac{u}{l_k},\qquad k = 1, 2 .
$$

With the trolley's position and speed there are six states.

- **Equal cables, $l_1 = l_2 = l$.** Subtract the two equations. The difference $\delta = \theta_1 - \theta_2$ obeys $\ddot\delta = -\tfrac{g}{l}\,\delta$. The input has dropped out. Whatever the trolley does, $\delta$ swings as if nobody were there. The rank test agrees: $\operatorname{rank}\mathcal R = 4$, not 6.
- **Unequal cables, $1.0$ m and $0.7$ m.** $\operatorname{rank}\mathcal R = 6$: one motor can steer everything.

<video controls loop muted playsinline width="100%" src="media/s31_control.mp4" poster="media/s31_control_poster.png"></video>

*First half, equal cables: three trolleys do three different things; the dials under them show $\theta_1 - \theta_2$, and the three needles move together. Second half, one cable shortened: the same three motions now give three different needles, and each trolley then brings both loads and itself to rest in 4 s with an input computed in Part 3.3.*

Two honest remarks about the robot. With equal cables, the three needles agree to within 0.2° on a swing of 20°, not exactly: the statement "the input drops out" belongs to the linear model, and violent shaking of the real pendulums does move the difference a little. And after steering to rest, the real loads are left swinging by at most 0.65°, where the linear model predicts zero.

> **Takeaway.** Equal cables: no trolley motion changes the difference. Controllability is a property of $A$ and $B$ together.
> ▶ Try it: [Controllability](https://realgaule.github.io/rob7104/lab4/#d31)

### What one sensor can see

**Question.** We measure only $y = Cx$. Can we work out the whole state from the history of $y$?

If yes, the system is **observable**. The test mirrors the first one:

$$
\mathcal O = \begin{pmatrix} C\\ CA\\ \vdots\\ CA^{n-1}\end{pmatrix},\qquad \text{observable} \iff \operatorname{rank}\mathcal O = n .
$$

$C$ says what the sensor reads now; $CA$, $CA^2$, … say what its derivatives reveal.

**The sensor.** Put a load cell on the parked trolley that measures the total sideways pull of the two cables. For small angles it reads $y = g\,(m_1\theta_1 + m_2\theta_2)$ (accurate to about 2% at 10°). With equal masses it measures the sum $\theta_1 + \theta_2$.

- **Equal cables.** The sum and the difference evolve separately, and the sensor reads only the sum. Two swings with the same sum but different differences give the same reading for all time. $\operatorname{rank}\mathcal O = 2$ of 4.
- **Unequal cables.** The two loads swing at different frequencies, both show up in the reading, and $\operatorname{rank}\mathcal O = 4$.

The swing that the motor could not move is the same one the sensor cannot see.

<video controls loop muted playsinline width="100%" src="media/s31_observe.mp4" poster="media/s31_observe_poster.png"></video>

*First part: two robots released differently (6° and 6°; 12° and 0°). With equal cables their load-cell readings coincide; with one cable shortened they separate. Second part: the loads are hidden (dark), and the green ghost is the estimate of a Kalman filter (Lecture 10) that sees only the noisy reading.*

With unequal cables the filter's estimate settles onto the true loads in about 2 s, and its error from then on is about 0.2° (root mean square). With equal cables it never finds the opposite swing: its error stays near 5°, and the filter knows it — its own uncertainty in that direction does not shrink.

> **Takeaway.** A sensor can only reconstruct what leaves a trace in its reading.
> ▶ Try it: [Observability](https://realgaule.github.io/rob7104/lab4/#d31b)

## 3.2 Two kicks cancel

**Question.** Back to the glass of water. Can the arm move it quickly and arrive with the surface flat?

**Idea.** Use Part 2.1 in reverse. A kick at time $0$ starts a slosh $A_1 e^{-\zeta\omega t}\sin(\omega_d t)$, where $\omega_d = \omega\sqrt{1-\zeta^2}$. A second kick, half a period later, starts a slosh of the opposite sign. If its size matches what is left of the first one, the two cancel from then on:

$$
t_2 = \frac{\pi}{\omega_d},\qquad \frac{A_2}{A_1} = e^{-\zeta\pi/\sqrt{1-\zeta^2}} .
$$

Because the system is linear and does not change with time, the same trick works for **any** motion: do a fraction $A_1/(A_1+A_2)$ of it now and the rest $t_2$ later. For the water, $t_2 = 0.138$ s and the two fractions are 0.516 and 0.484. This is called *input shaping*; it is how cranes and 3D printers avoid shaking their loads.

<video controls loop muted playsinline width="100%" src="media/s32_waiter.mp4" poster="media/s32_waiter_poster.png"></video>

*Both arms carry the glass 30 cm. Left: in one step. Right: in two half-steps, the second 0.138 s later. Slowed down about three times.*

| | Peak tilt | Slosh left after the hand stops |
|---|---|---|
| One step | 45° (spills: the rim is reached at 40.6°) | 29° |
| Two half-steps | 27° | 1.7° (the linear model predicts 0) |

The price is small and specific: the move takes 0.138 s longer, and the frequency of the slosh must be known in advance. Shaping removes the slosh that remains *after* the motion; it does not stop the surface from leaning back while the hand accelerates.

> **Takeaway.** Two kicks half a period apart cancel. Choosing the input well is already control.
> ▶ Try it: [Two kicks cancel](https://realgaule.github.io/rob7104/lab4/#d32): find the delay that leaves the water flat.

## 3.3 Steering to a target: rank says whether, the Gramian says how hard

**Question.** The rank test says a state can be reached. Which input reaches it, and at what cost?

Put $g = Bu$ in the formula of the lab and ask for $x(T) = x_f$:

$$
x_f - e^{AT}x_0 = \int_0^T e^{A(T-\tau)}B\,u(\tau)\,d\tau .
$$

Try an input of the form $u(\tau) = B^\top e^{A^\top(T-\tau)}\lambda$ with an unknown vector $\lambda$. The integral becomes $W(T)\lambda$, where

$$
W(T) = \int_0^T e^{As}BB^\top e^{A^\top s}\,ds
$$

is the **controllability Gramian**. So $\lambda = W(T)^{-1}\big(x_f - e^{AT}x_0\big)$, and

$$
u(t) = B^\top e^{A^\top(T-t)}\,W(T)^{-1}\big(x_f - e^{AT}x_0\big).
$$

Among all inputs that reach $x_f$ at time $T$, this one has the smallest energy $\int_0^T u^2\,dt$, and that smallest energy is $d^\top W(T)^{-1}d$ with $d = x_f - e^{AT}x_0$. $W(T)$ is invertible exactly when the system is controllable. (The same integral gave the covariance of a system driven by white noise in Lecture 10: where noise can push the state is where control can steer it.)

**The robot.** Three gantries carry the drone's two-part load 2 m and should arrive with it at rest.

<video controls loop muted playsinline width="100%" src="media/s33_transport.mp4" poster="media/s33_transport_poster.png"></video>

| Command | Arrives after | Energy $\int u^2$ | Sway left on the robot |
|---|---|---|---|
| Accelerate, then brake | 3.5 s | 1.49 | ±18.9 cm |
| Input shaping (3.2, once per mode) | 5.25 s | 0.59 | ±0.006 cm |
| Minimum energy for 3.5 s | 3.5 s | 1.22 | ±0.1 cm |

Read the table with care. "Minimum energy" means minimum *for the given time*: the shaped command uses less only because it takes 1.75 s longer, and the minimum-energy input for 5.25 s would need 0.35. The first command is also sensitive to timing: with a 5 s move its leftover sway happens to be 6 cm instead of 19 cm. The other two do not rely on luck.

**How hard is "controllable"?** Go back to the twin hoist of 3.1 and ask for the energy needed to stop a ±10° opposite swing in 4 s, as the second cable's length varies.

<img src="media/s33_energy_vs_ratio.png" width="100%">

At a length ratio of 0.7 it costs 0.88. At 0.95 it costs 66, and at 0.99 about 1800: the cost grows like one over the square of the distance to equal lengths. At exactly equal lengths no input can do it. The rank test jumps from 6 to 4 at a single point; the Gramian shows the approach.

<video controls loop muted playsinline width="100%" src="media/s33_reachable.mp4" poster="media/s33_reachable_poster.png"></video>

*The swings reachable from rest with one unit of input energy, as the allowed time grows. Horizontal: common swing; vertical: opposite swing. With a length ratio of 0.7 the set opens up in both directions. With 0.95 it stays a thin sliver: the opposite swing is reachable in principle and nearly unreachable in practice. These are statements about the linear model; they extend beyond small angles only as a guide.*

> **Takeaway.** Rank says whether a state can be reached; the Gramian says how much it costs.
> ▶ Try it: [Steering a crane](https://realgaule.github.io/rob7104/lab4/#d33): compare the three commands, then drive the trolley yourself.

---

## 4. Summary

$$
x(t) \;=\; \underbrace{e^{At}x_0}_{\text{Part 1}} \;+\; \int_0^t e^{A(t-\tau)}\,\underbrace{g(\tau)}_{\text{Part 2: given}\quad \text{Part 3: } Bu}\,d\tau
$$

| | The question | The tool | Seen on |
|---|---|---|---|
| **Passive** | What does the system do alone? | eigenvalues and modes of $A$; $e^{At}$; $\lVert e^{At}\rVert \le c\,e^{-at}$ | drone with slung load |
| **Forced** | What does a given forcing add? | the integral as a sum of kicks; the bound $c\gamma/a$; resonance | arm with a glass of water; shaken drone |
| **Controlled** | What can we make it do, and see? | rank of $[B\ AB\ \cdots]$ and of $[C;\ CA;\ \cdots]$; the Gramian $W(T)$ | gantry crane; arm with a glass of water |

**What this lab left out.** Every input in Part 3 was computed in advance and then played back. If someone bumps the load on the way, nothing reacts. Letting the input depend on the measured state, $u = -Kx$, turns the controlled system back into a passive one, $\dot x = (A - BK)x$, whose eigenvalues we get to place. That is feedback control, and it is the natural next step.

**Notes on the simulations.**

- The robots are simulated with MuJoCo. The arm is a Franka Emika Panda and the drone a Skydio X2 (models from MuJoCo Menagerie).
- The slung load and the crane loads are simulated with their full nonlinear equations. All inputs are designed on the linear model, and the leftover errors quoted in the text are those of the nonlinear robot.
- The water is an equivalent pendulum for the first sloshing mode, not a fluid simulation. The drone is treated as a support that can accelerate sideways; its tilting is not modelled.
- Hinge friction is viscous. The numbers in the tables are computed by the scripts that made the videos.